In [ ]:
import pandas as pd
import numpy as np
import seaborn.objects as so
import json
from pathlib import Path
from nps import io


## File Structure

in npe2api:
- public/
  - conda/ # specific files for each conda package
  - github/ # branch, issues, PRs, forks, stars, archived, last commit, etc <-- seems quite out of date
  - manifest/ # contains the napari.yaml manifests for each plugin
  - pypi/ # specific files
    - trove classifer
    - author
    - downloads?? maybe
    - release history
    - license
    - requirements
    - requires_python
    - recent release
- classifiers.json # active or withdraw, what pypi versions exist, and the name
- extended_summary.json # summary of:
  - name
  - current version
  - pypi versions
  - author
  - license
  - links
  - seems to exclude withdrawn or deleted plugins

So it looks like I will need to investigate classifiers and extended summary for some basic info, but then crawl specifically each pypi file for more detailed info (like python version, dependences, etc).

In [ ]:
NPE2API_DATA = Path("../npe2api/public")
OUTPUT_DIR = Path("reports")
OUTPUT_DIR.mkdir(exist_ok=True)

if NPE2API_DATA.exists():
    pypi_files = list(NPE2API_DATA.glob("pypi/*.json"))
    conda_files = list(NPE2API_DATA.glob("conda/*.json"))
    summary_df = io.load_json_as_dataframe(NPE2API_DATA / "extended_summary.json")
    classifiers_df = io.load_json_as_dataframe(NPE2API_DATA / "classifiers.json")



In [ ]:
status_cols = ['active', 'withdrawn', 'deleted']

classifiers_df[status_cols] = classifiers_df[status_cols].fillna(False).astype(bool)

classifiers_status = classifiers_df[status_cols].idxmax(axis=1)
# classifiers_status[classifiers_status != 'active']  # Show non-active packages
classifiers_status.value_counts()

In [ ]:
classifiers_df

Set up PyPi data

In [ ]:
pypi_data = {}
for file in pypi_files:
    plugin_name = file.stem
    pypi_data[plugin_name] = io.load_json(file)

pypi_df = pd.DataFrame.from_dict(pypi_data, orient='index')
pypi_df

In [ ]:
# add the classifier_df['status'] status to the pypi dataframe
pypi_df['status'] = classifiers_status
pypi_df

In [ ]:
pypi_info = pd.json_normalize(pypi_df['info'])
pypi_info